# TFT Local Training Pipeline

Local replacement for the Vertex AI training pipeline. Trains a TemporalFusionTransformer on 1-minute OHLCV data and uploads the best checkpoint to GCS.

**After training completes**, this notebook also recalculates the per-ticker bias correction factor used in `main.py` (`TICKER_BIAS_CORRECTION`).

## 0. Configuration

In [ ]:
# ===== CONFIGURE THESE =====
SYMBOL = "BTC"              # Ticker to train: SPY, GOOG, QQQ, TSLA, AAPL, BTC
POLYGON_API_KEY = "YOUR_KEY"  # Polygon.io API key (or set env POLYGON_API_KEY)

# Training hyperparameters
LOOKBACK_DAYS = 420         # Days of 1-min data to fetch
MAX_EPOCHS = 25
BATCH_SIZE = 64
LEARNING_RATE = 0.001
HIDDEN_SIZE = 64
ATTENTION_HEAD_SIZE = 4
DROPOUT = 0.1
HIDDEN_CONTINUOUS_SIZE = 32
PATIENCE = 10               # Early stopping patience

# Model architecture
MAX_ENCODER_LENGTH = 60
MAX_PREDICTION_LENGTH = 60

# GCS upload settings
GCS_BUCKET = "tft-for-trading-brains"
GCS_MODEL_PATH = f"tft_checkpoint_{SYMBOL}_latest.ckpt"

# Use GPU if available
import torch
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
print(f"Training on: {ACCELERATOR}")
print(f"Symbol: {SYMBOL}, Lookback: {LOOKBACK_DAYS} days, Epochs: {MAX_EPOCHS}")

## 1. Install / Import Dependencies

In [ ]:
import os
import time
import gc
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, GroupNormalizer, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss, MultiHorizonMetric
from google.cloud import storage, bigquery
import requests

# Resolve Polygon API key
if POLYGON_API_KEY == "YOUR_KEY":
    POLYGON_API_KEY = os.environ.get("POLYGON_API_KEY", "")
    assert POLYGON_API_KEY, "Set POLYGON_API_KEY in the config cell or as an environment variable"

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")

## 2. Custom Loss Function

In [ ]:
class DirectionAwareQuantileLoss(MultiHorizonMetric):
    """
    Composite loss: QuantileLoss + direction penalty.
    Penalizes predictions where the predicted direction of change
    disagrees with the actual direction of change.
    """

    def __init__(
        self,
        quantile_weight: float = 0.7,
        direction_weight: float = 0.3,
        quantiles: list = [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98],
        **kwargs,
    ):
        super().__init__(quantiles=quantiles, **kwargs)
        self.quantile_weight = quantile_weight
        self.direction_weight = direction_weight
        self.quantiles = quantiles

    def loss(self, y_pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        losses = []
        for i, q in enumerate(self.quantiles):
            errors = target - y_pred[..., i]
            q_loss = torch.max((q - 1) * errors, q * errors)
            losses.append(q_loss.unsqueeze(-1))
        quantile_loss = torch.cat(losses, dim=-1).mean(dim=-1)

        median_idx = len(self.quantiles) // 2
        pred_median = y_pred[..., median_idx]

        pred_change = pred_median[:, 1:] - pred_median[:, :-1]
        actual_change = target[:, 1:] - target[:, :-1]

        direction_agreement = torch.tanh(pred_change * 10) * torch.tanh(actual_change * 10)
        direction_penalty = torch.clamp(1.0 - direction_agreement, min=0.0) / 2.0

        pad = torch.zeros_like(direction_penalty[:, :1])
        direction_penalty = torch.cat([pad, direction_penalty], dim=1)

        return self.quantile_weight * quantile_loss + self.direction_weight * direction_penalty

## 3. Data Fetching (Polygon.io)

In [ ]:
def fetch_polygon_1min_data(symbol: str, start_date: str, end_date: str, api_key: str, max_retries: int = 10) -> pd.DataFrame:
    """Fetch 1-minute OHLCV data from Polygon.io with pagination and retry."""
    # Map simple ticker to Polygon format for crypto
    POLYGON_SYMBOL_MAP = {
        'BTC': 'X:BTCUSD',
        'ETH': 'X:ETHUSD',
    }
    polygon_symbol = POLYGON_SYMBOL_MAP.get(symbol, symbol)
    
    all_data = []
    url = f"https://api.polygon.io/v2/aggs/ticker/{polygon_symbol}/range/1/minute/{start_date}/{end_date}"
    params = {"adjusted": "true", "sort": "asc", "limit": 50000, "apiKey": api_key}

    print(f"Fetching {polygon_symbol} data from {start_date} to {end_date}...")

    while url:
        for attempt in range(1, max_retries + 1):
            try:
                response = requests.get(url, params=params, timeout=60)
                data = response.json()
            except (requests.RequestException, ValueError) as e:
                print(f"  Request error (attempt {attempt}/{max_retries}): {e}")
                if attempt < max_retries:
                    wait = min(3 ** attempt, 120)
                    print(f"  Retrying in {wait}s...")
                    time.sleep(wait)
                    continue
                raise

            if data.get("status") in ("OK", "DELAYED") and "results" in data:
                break

            print(f"  API response: {data.get('status', 'unknown')} - {data.get('message', '')} (attempt {attempt}/{max_retries})")
            if attempt < max_retries:
                wait = min(3 ** attempt, 120)
                print(f"  Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  Giving up after {max_retries} attempts.")
                break
        else:
            break

        if data.get("status") not in ("OK", "DELAYED") or "results" not in data:
            break

        all_data.extend(data["results"])
        print(f"  Fetched {len(all_data):,} bars so far...")

        next_url = data.get("next_url")
        if next_url:
            url = next_url
            params = {"apiKey": api_key}
            time.sleep(15)  # Stay within free-tier rate limit
        else:
            url = None

    if not all_data:
        raise ValueError(f"No data returned from Polygon for {polygon_symbol}")

    df = pd.DataFrame(all_data)
    df["timestamp"] = pd.to_datetime(df["t"], unit="ms", utc=True)
    df["timestamp"] = df["timestamp"].dt.tz_convert("America/New_York").dt.tz_localize(None)
    df = df.rename(columns={"o": "open", "h": "high", "l": "low", "c": "close", "v": "volume"})
    df = df[["timestamp", "open", "high", "low", "close", "volume"]]
    df = df.sort_values("timestamp").reset_index(drop=True)

    # For crypto: volume may be 0; fill with 1.0 to avoid div-by-zero
    if df['volume'].sum() == 0:
        df['volume'] = 1.0

    print(f"Total: {len(df):,} 1-minute bars from {df['timestamp'].min()} to {df['timestamp'].max()}")
    return df

In [ ]:
# Fetch data
end_date = datetime.utcnow().strftime("%Y-%m-%d")
start_date = (datetime.utcnow() - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")

df_raw = fetch_polygon_1min_data(SYMBOL, start_date, end_date, POLYGON_API_KEY)
df_raw.tail()

## 4. Feature Engineering

In [ ]:
def calculate_all_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate all 76 features required for the TFT model.
    Input df must have: timestamp, open, high, low, close, volume
    """
    df = df.copy()

    # 1. RETURNS (5 features)
    df['returns_1m'] = df['close'].pct_change(1)
    df['returns_5m'] = df['close'].pct_change(5)
    df['returns_15m'] = df['close'].pct_change(15)
    df['returns_30m'] = df['close'].pct_change(30)
    df['returns_60m'] = df['close'].pct_change(60)

    # 2. PRICE RATIOS (6 features)
    df['high_low_ratio'] = df['high'] / df['low']
    df['close_open_ratio'] = df['close'] / df['open']
    df['high_close_ratio'] = df['high'] / df['close']
    df['low_close_ratio'] = df['low'] / df['close']
    df['upper_shadow'] = (df['high'] - np.maximum(df['open'], df['close'])) / (df['high'] - df['low'] + 1e-10)
    df['lower_shadow'] = (np.minimum(df['open'], df['close']) - df['low']) / (df['high'] - df['low'] + 1e-10)

    # 3. SMA FEATURES (11 features)
    for period in [5, 10, 20, 30]:
        sma = df['close'].rolling(window=period).mean()
        df[f'sma_{period}_slope'] = sma.pct_change()
        df[f'close_to_sma_{period}'] = (df['close'] - sma) / sma

    sma_60 = df['close'].rolling(window=60).mean()
    df['close_to_sma_60'] = (df['close'] - sma_60) / sma_60

    sma_120 = df['close'].rolling(window=120).mean()
    df['sma_120_slope'] = sma_120.pct_change()
    df['close_to_sma_120'] = (df['close'] - sma_120) / sma_120

    # 4. MACD (2 features)
    ema_12 = df['close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema_12 - ema_26
    signal_line = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_histogram'] = df['macd'] - signal_line

    # 5. VOLATILITY (4 features)
    df['volatility_5'] = df['returns_1m'].rolling(window=5).std()
    df['volatility_10'] = df['returns_1m'].rolling(window=10).std()
    df['volatility_20'] = df['returns_1m'].rolling(window=20).std()
    df['volatility_60'] = df['returns_1m'].rolling(window=60).std()

    # 6. ATR (2 features)
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    tr = np.maximum(high_low, np.maximum(high_close, low_close))
    df['atr_14'] = tr.rolling(window=14).mean()
    df['atr_60'] = tr.rolling(window=60).mean()

    # 7. BOLLINGER BANDS (4 features)
    for period in [20, 60]:
        sma = df['close'].rolling(window=period).mean()
        std = df['close'].rolling(window=period).std()
        df[f'bb_std_{period}'] = std
        df[f'bb_position_{period}'] = (df['close'] - sma) / (2 * std + 1e-10)

    # 8. RSI (3 features)
    def calc_rsi(series, period):
        delta = series.diff()
        gain = delta.where(delta > 0, 0).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / (loss + 1e-10)
        return 100 - (100 / (1 + rs))

    df['rsi_14'] = calc_rsi(df['close'], 14)
    df['rsi_20'] = calc_rsi(df['close'], 20)
    df['rsi_60'] = calc_rsi(df['close'], 60)

    # 9. STOCHASTIC (3 features)
    for period in [14, 60]:
        lowest_low = df['low'].rolling(window=period).min()
        highest_high = df['high'].rolling(window=period).max()
        df[f'stoch_k_{period}'] = 100 * (df['close'] - lowest_low) / (highest_high - lowest_low + 1e-10)
    df['stoch_d_14'] = df['stoch_k_14'].rolling(window=3).mean()

    # 10. RATE OF CHANGE (2 features)
    df['roc_10'] = df['close'].pct_change(10) * 100
    df['roc_20'] = df['close'].pct_change(20) * 100

    # 11. VOLUME INDICATORS (12 features)
    df['volume_change'] = df['volume'].pct_change(1)
    df['volume_change_5m'] = df['volume'].pct_change(5)

    for period in [5, 10, 20, 60]:
        df[f'volume_sma_{period}'] = df['volume'].rolling(window=period).mean()
        df[f'volume_ratio_{period}'] = df['volume'] / (df[f'volume_sma_{period}'] + 1e-10)

    # 12. OBV, VPT, MFI (5 features)
    obv = np.where(df['close'] > df['close'].shift(), df['volume'],
                   np.where(df['close'] < df['close'].shift(), -df['volume'], 0))
    df['obv'] = np.cumsum(obv)
    obv_sma = pd.Series(df['obv']).rolling(window=20).mean()
    df['obv_ratio'] = df['obv'] / (obv_sma + 1e-10)

    df['vpt'] = (df['volume'] * df['close'].pct_change()).cumsum()

    def calc_mfi(df, period):
        typical_price = (df['high'] + df['low'] + df['close']) / 3
        money_flow = typical_price * df['volume']
        positive_flow = money_flow.where(typical_price > typical_price.shift(), 0).rolling(window=period).sum()
        negative_flow = money_flow.where(typical_price < typical_price.shift(), 0).rolling(window=period).sum()
        mfi = 100 - (100 / (1 + positive_flow / (negative_flow + 1e-10)))
        return mfi

    df['mfi_14'] = calc_mfi(df, 14)
    df['mfi_60'] = calc_mfi(df, 60)

    # 13. VWAP RATIOS (2 features)
    vwap_20 = (df['volume'] * df['close']).rolling(window=20).sum() / (df['volume'].rolling(window=20).sum() + 1e-10)
    vwap_60 = (df['volume'] * df['close']).rolling(window=60).sum() / (df['volume'].rolling(window=60).sum() + 1e-10)
    df['close_to_vwap_20'] = (df['close'] - vwap_20) / vwap_20
    df['close_to_vwap_60'] = (df['close'] - vwap_60) / vwap_60

    # 14. TIME FEATURES (10 features)
    df['hour'] = df['timestamp'].dt.hour
    df['minute'] = df['timestamp'].dt.minute
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_morning'] = ((df['hour'] >= 9) & (df['hour'] < 12)).astype(int)
    df['is_afternoon'] = ((df['hour'] >= 12) & (df['hour'] < 16)).astype(int)

    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    # 15. VOLUME LAGS (5 features)
    df['volume_lag_1'] = df['volume'].shift(1)
    df['volume_lag_5'] = df['volume'].shift(5)
    df['volume_lag_15'] = df['volume'].shift(15)
    df['volume_lag_30'] = df['volume'].shift(30)
    df['volume_lag_60'] = df['volume'].shift(60)

    # 16. TARGET VARIABLES (2 features)
    df['target_close_60m'] = df['close'].shift(-60)
    df['target_return_60m'] = df['close'].pct_change(60).shift(-60)

    return df

In [ ]:
def remove_highly_correlated_features(df: pd.DataFrame, threshold: float = 0.95):
    """Remove features with pairwise correlation > threshold."""
    exclude_cols = [
        "timestamp", "time_idx", "group",
        "target_close_60m", "target_return_60m",
        "open", "high", "low", "close", "volume", "vwap",
    ]
    feature_cols = [
        c for c in df.columns
        if c not in exclude_cols and df[c].dtype in ["float64", "float32", "int64"]
    ]

    df_clean = df[feature_cols].dropna()
    corr_matrix = df_clean.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]

    print(f"Correlation filtering (threshold={threshold}):")
    print(f"  Features before: {len(feature_cols)}")
    print(f"  Features dropped: {len(to_drop)} — {to_drop[:10]}{'...' if len(to_drop) > 10 else ''}")
    print(f"  Features remaining: {len(feature_cols) - len(to_drop)}")

    return df.drop(columns=to_drop), to_drop


def prepare_dataset(df: pd.DataFrame):
    """Run feature engineering and prepare the TFT-ready DataFrame."""
    # Calculate VWAP
    if df['volume'].sum() > 0:
        df["vwap"] = (df["volume"] * (df["high"] + df["low"] + df["close"]) / 3).cumsum() / df["volume"].cumsum()
    else:
        df["vwap"] = (df["high"] + df["low"] + df["close"]) / 3

    # Feature engineering
    df_feat = calculate_all_features(df)

    # Drop rows with NaN from indicators
    target_cols = ["target_close_60m", "target_return_60m"]
    non_target = [c for c in df_feat.columns if c not in target_cols]
    df_feat = df_feat.dropna(subset=non_target)
    df_feat = df_feat.dropna(subset=target_cols)
    df_feat = df_feat.reset_index(drop=True)

    # Correlation filtering
    df_feat, dropped_cols = remove_highly_correlated_features(df_feat, threshold=0.95)

    # Add required columns
    df_feat["time_idx"] = range(len(df_feat))
    df_feat["group"] = "default"

    # Clean infinities
    numeric_cols = df_feat.select_dtypes(include=[np.number]).columns.tolist()
    df_feat[numeric_cols] = df_feat[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df_feat[numeric_cols] = df_feat[numeric_cols].ffill().bfill()

    print(f"\nPrepared dataset: {len(df_feat):,} rows, {len(df_feat.columns)} columns")
    return df_feat

In [ ]:
# Run feature engineering
df_tft = prepare_dataset(df_raw)

# Save metadata
data_start = str(df_raw['timestamp'].min())
data_end = str(df_raw['timestamp'].max())
print(f"Data range: {data_start} to {data_end}")

# Free raw data
del df_raw
gc.collect()

## 5. Build TimeSeriesDataSet

In [ ]:
def build_feature_lists(df: pd.DataFrame):
    """Derive the time-varying known and unknown reals lists."""
    time_varying_known_reals = ["hour_sin", "hour_cos", "minute_sin", "minute_cos", "day_sin", "day_cos"]
    time_varying_known_reals = [c for c in time_varying_known_reals if c in df.columns]

    exclude = [
        "timestamp", "time_idx", "group",
        "target_close_60m", "target_return_60m",
        "hour", "minute", "day_of_week",
        "is_morning", "is_afternoon",
    ] + time_varying_known_reals

    time_varying_unknown_reals = [c for c in df.columns if c not in exclude]

    if "close" in time_varying_unknown_reals:
        time_varying_unknown_reals.remove("close")
    time_varying_unknown_reals = ["close"] + time_varying_unknown_reals

    print(f"Time-varying known reals: {len(time_varying_known_reals)}")
    print(f"Time-varying unknown reals: {len(time_varying_unknown_reals)}")
    return time_varying_known_reals, time_varying_unknown_reals


time_varying_known_reals, time_varying_unknown_reals = build_feature_lists(df_tft)

# Train/validation split (80/20)
training_cutoff = int(len(df_tft) * 0.8)

print(f"\nDataset size: {len(df_tft):,} rows")
print(f"Training cutoff: time_idx={training_cutoff}")
print(f"Training rows: {len(df_tft[df_tft['time_idx'] <= training_cutoff]):,}")
print(f"Validation rows: {len(df_tft[df_tft['time_idx'] > training_cutoff]):,}")

In [ ]:
training = TimeSeriesDataSet(
    df_tft[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="close",
    group_ids=["group"],
    min_encoder_length=MAX_ENCODER_LENGTH // 2,
    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=["group"],
    time_varying_known_categoricals=[],
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_categoricals=[],
    time_varying_unknown_reals=time_varying_unknown_reals,
    target_normalizer=GroupNormalizer(
        groups=["group"],
        transformation="softplus",
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    df_tft[lambda x: x.time_idx > training_cutoff],
    predict=False,
)

training_rows = len(training)
validation_rows = len(validation)
print(f"Training samples: {training_rows:,}, Validation samples: {validation_rows:,}")

# Free DataFrame
del df_tft
gc.collect()

# Create dataloaders
train_dataloader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
val_dataloader = training.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)

## 6. Build & Train Model

In [ ]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=LEARNING_RATE,
    hidden_size=HIDDEN_SIZE,
    attention_head_size=ATTENTION_HEAD_SIZE,
    dropout=DROPOUT,
    hidden_continuous_size=HIDDEN_CONTINUOUS_SIZE,
    output_size=7,  # 7 quantiles
    loss=DirectionAwareQuantileLoss(quantile_weight=0.7, direction_weight=0.3),
    reduce_on_plateau_patience=4,
)
print(f"Model parameters: {tft.size() / 1e3:.1f}k")

In [ ]:
# Callbacks
checkpoint_dir = "checkpoints/training"
os.makedirs(checkpoint_dir, exist_ok=True)

early_stop = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=PATIENCE, verbose=True, mode="min")
lr_monitor = LearningRateMonitor(logging_interval="step")
checkpoint_cb = ModelCheckpoint(
    dirpath=checkpoint_dir,
    monitor="val_loss",
    filename=f"tft-{SYMBOL}" + "-{epoch:02d}-{val_loss:.4f}",
    save_top_k=1,
    mode="min",
)

# Trainer
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    callbacks=[early_stop, lr_monitor, checkpoint_cb],
    enable_model_summary=True,
    accelerator=ACCELERATOR,
    gradient_clip_val=0.1,
)

print(f"Starting training on {ACCELERATOR}...")
trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

In [ ]:
best_model_path = checkpoint_cb.best_model_path
best_val_loss = float(checkpoint_cb.best_model_score)
print(f"Best model: {best_model_path}")
print(f"Best val_loss: {best_val_loss:.4f}")
print(f"Epochs trained: {trainer.current_epoch + 1}")

## 7. Evaluate on Validation Set

In [ ]:
# Load best checkpoint
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path, strict=False)

# Get predictions on validation set
predictions = best_tft.predict(val_dataloader, return_x=True)

if predictions.output.dim() == 3:
    pred_values = predictions.output[:, :, 3]  # median quantile
else:
    pred_values = predictions.output

actuals = predictions.x["decoder_target"]
if actuals.dim() == 3:
    actuals = actuals[:, :, 0]

pred_np = pred_values.detach().cpu().numpy().flatten()
actual_np = actuals.detach().cpu().numpy().flatten()

# Metrics
errors = actual_np - pred_np
mae = float(np.mean(np.abs(errors)))
rmse = float(np.sqrt(np.mean(errors ** 2)))
mask = actual_np != 0
mape = float(np.mean(np.abs(errors[mask] / actual_np[mask])) * 100) if mask.any() else None
ss_res = np.sum(errors ** 2)
ss_tot = np.sum((actual_np - np.mean(actual_np)) ** 2)
r_squared = float(1 - ss_res / ss_tot) if ss_tot != 0 else None

print(f"Validation Metrics:")
print(f"  MAE:   {mae:.4f}")
print(f"  RMSE:  {rmse:.4f}")
print(f"  MAPE:  {mape:.4f}%" if mape else "  MAPE:  N/A")
print(f"  R²:    {r_squared:.4f}" if r_squared else "  R²:    N/A")

## 8. Calculate Bias Correction Factor

This computes the directional bias of the model on the validation set.  
The resulting value should be set in `main.py` → `TICKER_BIAS_CORRECTION['{SYMBOL}']`.

**Formula**: `bias = mean(actual_return) - mean(predicted_return)`  
A positive bias means the model is too bearish; a negative bias means too bullish.

In [ ]:
# Compute bias as return difference (fraction, not %)
# Use 15-step ahead predictions vs actuals (most relevant for trading signals)
# We compare predicted % change vs actual % change

# Get base prices (encoder last step)
encoder_target = predictions.x["encoder_target"]
if encoder_target.dim() == 3:
    encoder_target = encoder_target[:, :, 0]
base_prices = encoder_target[:, -1].detach().cpu().numpy()  # last encoder step

# Predicted and actual at each horizon
pred_all = pred_values.detach().cpu().numpy()  # (batch, horizon)
actual_all = actuals.detach().cpu().numpy()     # (batch, horizon)

# Calculate returns relative to base price
pred_returns = (pred_all - base_prices[:, None]) / base_prices[:, None]
actual_returns = (actual_all - base_prices[:, None]) / base_prices[:, None]

# Mean return at each horizon
mean_pred = pred_returns.mean(axis=0)
mean_actual = actual_returns.mean(axis=0)

# Overall bias (average across all horizons)
bias_per_horizon = mean_actual - mean_pred
overall_bias = float(bias_per_horizon.mean())

print(f"=" * 50)
print(f"BIAS ANALYSIS for {SYMBOL}")
print(f"=" * 50)
print(f"Mean predicted return: {mean_pred.mean()*100:.4f}%")
print(f"Mean actual return:    {mean_actual.mean()*100:.4f}%")
print(f"Bias (actual - pred):  {overall_bias*100:.4f}%")
print(f"")
print(f"Per-horizon bias (15m, 30m, 45m, 60m):")
for step, label in [(14, '15m'), (29, '30m'), (44, '45m'), (59, '60m')]:
    if step < len(bias_per_horizon):
        print(f"  {label}: {bias_per_horizon[step]*100:.4f}%")
print(f"")
print(f">>> Update main.py: TICKER_BIAS_CORRECTION['{SYMBOL}'] = {overall_bias:.6f}")
print(f"    (positive = model too bearish, negative = model too bullish)")

In [ ]:
# Direction accuracy analysis
# How often does the model correctly predict up vs down?
pred_direction = np.sign(pred_returns)
actual_direction = np.sign(actual_returns)

# Overall direction accuracy
correct = (pred_direction == actual_direction)
overall_acc = correct.mean()

# Accuracy when model predicts UP
up_mask = pred_direction > 0
up_acc = correct[up_mask].mean() if up_mask.any() else 0
up_count = up_mask.sum()

# Accuracy when model predicts DOWN
down_mask = pred_direction < 0
down_acc = correct[down_mask].mean() if down_mask.any() else 0
down_count = down_mask.sum()

print(f"Direction Analysis:")
print(f"  Overall accuracy: {overall_acc*100:.1f}%")
print(f"  UP calls:   {up_count:,} ({up_count/(up_count+down_count)*100:.1f}%) — accuracy: {up_acc*100:.1f}%")
print(f"  DOWN calls: {down_count:,} ({down_count/(up_count+down_count)*100:.1f}%) — accuracy: {down_acc*100:.1f}%")
print(f"")
print(f"  Skew ratio (down/up): {down_count/max(up_count,1):.1f}:1")

## 9. Upload to GCS

In [ ]:
print(f"Uploading checkpoint to gs://{GCS_BUCKET}/{GCS_MODEL_PATH}...")
storage_client = storage.Client()
bucket = storage_client.bucket(GCS_BUCKET)

# Upload as latest
blob = bucket.blob(GCS_MODEL_PATH)
blob.upload_from_filename(best_model_path)

# Upload timestamped copy
timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
timestamped_path = f"tft_checkpoint_{SYMBOL}_{timestamp}.ckpt"
blob_ts = bucket.blob(timestamped_path)
blob_ts.upload_from_filename(best_model_path)

print(f"✓ Uploaded: gs://{GCS_BUCKET}/{GCS_MODEL_PATH}")
print(f"✓ Uploaded: gs://{GCS_BUCKET}/{timestamped_path}")

## 10. Log Metrics to BigQuery

In [ ]:
bq_client = bigquery.Client(project="trading-brains")
table_id = "trading-brains.tft_predictions.retraining_eval_logs"

row = {
    "run_timestamp": datetime.utcnow().isoformat(),
    "symbol": SYMBOL,
    "best_val_loss": round(best_val_loss, 6),
    "mae": round(mae, 6),
    "rmse": round(rmse, 6),
    "mape": round(mape, 6) if mape else None,
    "r_squared": round(r_squared, 6) if r_squared else None,
    "epochs_trained": trainer.current_epoch + 1,
    "max_epochs": MAX_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "hidden_size": HIDDEN_SIZE,
    "attention_head_size": ATTENTION_HEAD_SIZE,
    "dropout": DROPOUT,
    "patience": PATIENCE,
    "lookback_days": LOOKBACK_DAYS,
    "data_start_date": data_start,
    "data_end_date": data_end,
    "training_rows": training_rows,
    "validation_rows": validation_rows,
    "model_gcs_path": f"gs://{GCS_BUCKET}/{GCS_MODEL_PATH}",
    "bias_correction": overall_bias,
}

errors_bq = bq_client.insert_rows_json(table_id, [row])
if errors_bq:
    print(f"BigQuery insert errors: {errors_bq}")
else:
    print(f"✓ Metrics logged to {table_id}")

## 11. Summary & Next Steps

In [ ]:
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Symbol:         {SYMBOL}")
print(f"  Epochs:         {trainer.current_epoch + 1}")
print(f"  Best val_loss:  {best_val_loss:.4f}")
print(f"  MAE:            {mae:.4f}")
print(f"  R²:             {r_squared:.4f}" if r_squared else "  R²:  N/A")
print(f"  Data range:     {data_start} → {data_end}")
print(f"  Model:          gs://{GCS_BUCKET}/{GCS_MODEL_PATH}")
print(f"")
print(f"  BIAS CORRECTION: {overall_bias:.6f}")
print(f"")
print(f"ACTION REQUIRED:")
print(f"  Update main.py → TICKER_BIAS_CORRECTION['{SYMBOL}'] = {overall_bias:.6f}")
print(f"  Then rebuild & redeploy the prediction service.")
print("=" * 60)